# A3.5 · Validating what comes back

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**.

| | |
|---|---|
| Tools used | Inspect |

## What this lesson is

**What it covers.** Pass a fabricated claim through a schema check and then through a ground-truth verifier.

**Why a security engineer needs it.** An unverified claim becomes a shared premise, and a peer message is trusted more than a document it is no safer than. The control it builds is: schema validation plus an independent verifier before any claim propagates.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A tool result re-enters the context as a fact. So does a peer's message. A schema check proves the shape is right and says nothing at all about whether the claim inside it is true.

> **At CyberTravels.** The payments API returns `{"status":"refunded"}`. That is a valid shape and it is not evidence the money moved, and the advisor's hotel recommendation is the same problem in prose. R2.

## 2 · The framework

```
   tool result / peer message
            |
            v
   +------------------+   shape is valid, claim may be false
   |  schema check    |   {"status":"ok","rows":0}  <- conforms perfectly
   +--------+---------+
            v
   +------------------+   the claim, checked against something independent
   |    verifier      |   did the row actually appear in the database?
   +------------------+

   conformance is about the serialiser. accuracy is the expensive part.
```

**Mitigates: T5 Cascading Hallucination · T12 Communication Poisoning · T7 Misaligned Behaviour.**

Everything that comes back into the context is an input: tool results, peer
messages, retrieved documents, a sub-agent's summary. A1.10 and A1.12 both
happened because those inputs were trusted in proportion to how internal they
looked rather than to how checked they were.

Two different checks, and conflating them is the mistake:

**Schema validation** asks *is this the right shape*. Cheap, mechanical, catches
malformed input and injection through a field that was supposed to be an
integer. It is necessary and it proves nothing about truth — a perfectly-formed
JSON object can assert anything.

**Verification** asks *is this claim true, according to something that is true
independently of the agent*. A test that passes. A query whose result you can
re-run. A file that exists. A signature that checks.

The rule that follows: **a claim may not propagate past the hop that produced
it without a verification result attached.** Not "was it plausible" — was it
checked, by what, and what did that return.

That single field is what stops A1.12's cascade, because confidence can no
longer rise as evidence disappears: the evidence field travels with the claim,
and an empty one is visible at every hop.

It is also the answer to A1.16, which is why "ask the model whether it
succeeded" is not a verifier — it is the same component grading its own work.

> **What this control closes.**
>
> Stops a claim propagating without evidence attached. Schema validity is not truth: a well-formed object can assert anything.

## 3 · The check, as a skill

`{"status":"refunded"}` is well-formed and may be false. The skill checks four returns twice — schema, then an independent oracle — and confirms that a claim with no oracle stops as `unverifiable` rather than quietly becoming true.

In [ ]:
# skills/runtime/tool-return-validation-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: tool-return-validation-check
description: >-
  Check what a tool's return value is validated against before the agent acts on
  it — schema, then an independent oracle — and confirm that an unverifiable
  claim stops rather than propagating. Use when a tool's output becomes an
  agent's belief, or when reviewing multi-hop reasoning.
allowed-tools: Read, Grep, Glob
---

# Well-formed is not true

A tool return is untrusted input that arrives wearing the tool's authority. Two
checks are needed and they catch different things: a **schema** catches
malformed, and an **oracle** catches confidently wrong. Systems usually have the
first and treat it as if it were the second.

## When to use this

Any agent that acts on what a tool told it, and any pipeline where one step's
output is the next step's premise.

## Procedure

**1 — Define the schema per tool return.** Types and required fields. This is
the cheap check and it should be automatic; a return that fails it never
reaches the model.

**2 — Identify the oracle for each claim type.** Something independent that can
say true or false: a CVE database, a build, a test run, a second source. Not
another model — a model checking a model measures agreement, not truth.

**3 — Run the four cases.** Schema-perfect and true; schema-perfect and false;
schema-perfect with **no oracle available**; malformed. All four have to be
distinguishable in the output.

**4 — Make "unverifiable" a terminal state.** The third case is the one that
matters. A claim with no oracle must stop as `unverifiable` rather than
defaulting to true — silent promotion is how a hedge becomes a fact three hops
later.

**5 — Confirm only verified claims propagate.** Follow each case for several
hops and record which survive. The unverified ones surviving is the finding.

## Output contract

```json
{
  "tools": [{"name": "str", "schema": true, "oracle": "str|null"}],
  "cases": [{"claim": "str", "schema_ok": true, "oracle_verdict": "true|false|unavailable",
             "state": "verified|refuted|unverifiable|malformed", "propagated": false}],
  "unverifiable_is_terminal": true
}
```

## Failure modes

- **Treating schema conformance as verification.** It is a statement about the
  serialiser.
- **Using a model as the oracle.** Two models agreeing is not evidence.
- **Defaulting unverifiable to true** because the pipeline needs a value. That
  default is the defect.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/runtime/tool-return-validation-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/runtime/tool-return-validation-check/scripts/tool_return_validation_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check four tool returns against a schema and against an oracle, and propagate only what verified.

This is the executable half of the `tool-return-validation-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

SCHEMA = {"claim": str, "confidence": float, "verified_by": (str, type(None))}

def schema_ok(msg):
    return all(k in msg and isinstance(msg[k], t) for k, t in SCHEMA.items())

GROUND_TRUTH = {"libfoo has no known CVEs": False,      # it has one
                "test_login passes": True}

def verify(claim):
    """Independent: reads ground truth, not the sender's opinion."""
    if claim not in GROUND_TRUTH:
        return None, "no oracle for this claim"
    return GROUND_TRUTH[claim], "checked against the advisory database"

def propagate(msg, hops=3):
    """A claim may not travel without a verification result attached."""
    if not schema_ok(msg):
        return {"stopped": "malformed"}
    result, how = verify(msg["claim"])
    if result is None:
        return {"stopped": "unverifiable", "claim": msg["claim"], "why": how}
    if result is False:
        return {"stopped": "refuted", "claim": msg["claim"], "by": how}
    return {"propagated": msg["claim"], "verified_by": how, "hops": hops}

MESSAGES = [
 {"claim": "libfoo has no known CVEs", "confidence": 0.9, "verified_by": None},
 {"claim": "test_login passes",        "confidence": 0.5, "verified_by": None},
 {"claim": "the refund was approved",  "confidence": 0.99, "verified_by": None},
 {"claim": "libfoo is fine",           "confidence": "high", "verified_by": None},
]
for m in MESSAGES:
    print(f"   schema_ok={str(schema_ok(m)):5s} -> {propagate(m)}")

print()
print("The first message is schema-perfect and confident and false. Schema")
print("validation passed it; the oracle refuted it.")
print()
print("The third is unverifiable - no oracle exists. That is a legitimate")
print("outcome and it must not silently become 'true'. It stops here with a")
print("reason, which is what A1.12's cascade never had.")
assert propagate(MESSAGES[0])["stopped"] == "refuted"
assert propagate(MESSAGES[2])["stopped"] == "unverifiable"
assert "propagated" in propagate(MESSAGES[1])

## What you just proved

Four messages are checked twice. A schema-perfect, high-confidence claim is refuted by the oracle; a claim with no oracle stops with `unverifiable` rather than silently becoming true; a malformed message is caught by the schema; and only the verified claim propagates.

## Your turn

Find one place a sub-agent's output becomes another agent's input and ask what oracle checks it. If the answer is the model's own confidence, that is the component grading its own work.

---

**Next → [A3.6 · Human approval that survives volume](https://spbreed.github.io/cyber-commons/lessons/A3.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*